## Importacion de librerias


In [2]:
import numpy as np
import pandas as pd
import math
import time

# Fijar semilla global opcional (para reproducibilidad total)
np.random.seed(42)

## Definicion de las funciones objetivo y sus dominios

In [3]:
def sphere(x):
    return np.sum(x**2)

def zakharov(x):
    n = len(x)
    sum1 = np.sum(x**2)
    sum2 = np.sum([0.5 * (i + 1) * x[i] for i in range(n)])
    return sum1 + sum2**2 + sum2**4

def rosenbrock(x):
    n = len(x)
    return np.sum([100 * (x[i+1] - x[i]**2)**2 + (x[i] - 1)**2 for i in range(n - 1)])

def schwefel(x):
    d = len(x)
    return 418.9829 * d - np.sum(x * np.sin(np.sqrt(np.abs(x))))

def michalewicz(x, m=10):
    n = len(x)
    return -np.sum([np.sin(x[i]) * (np.sin((i + 1) * x[i]**2 / np.pi))**(2 * m) for i in range(n)])

# Diccionario para acceder fácilmente a los dominios y funciones
benchmark_funcs = {
    'Sphere':      {'func': sphere,      'bounds': [-5.12, 5.12]},
    'Zakharov':    {'func': zakharov,    'bounds': [-5.0, 10.0]},
    'Rosenbrock':  {'func': rosenbrock,  'bounds': [-5.0, 10.0]},
    'Schwefel':    {'func': schwefel,    'bounds': [-500.0, 500.0]},
    'Michalewicz': {'func': michalewicz, 'bounds': [0.0, np.pi]}
}

## Decodificacion del algoritmo genetico

In [4]:
def decode_chromosome(chromosome, dim, bounds, bits_per_var=8):
    """Transforma un arreglo binario a un vector de valores reales."""
    real_values = np.zeros(dim)
    l_min, l_max = bounds
    max_decimal = (2**bits_per_var) - 1
    
    for i in range(dim):
        # Extraer los 8 bits de la variable i
        sub_chromosome = chromosome[i*bits_per_var : (i+1)*bits_per_var]
        # Convertir a decimal
        decimal_val = int("".join(sub_chromosome.astype(str)), 2)
        # Mapeo a valor real
        real_values[i] = l_min + decimal_val * ((l_max - l_min) / max_decimal)
        
    return real_values

## Asignacion de operadores del AG

In [5]:
def rank_selection(population_fitness, num_parents):
    """Selección de padres basada en ranking."""
    pop_size = len(population_fitness)
    # Ordenar índices del mejor (menor fitness) al peor
    ranked_indices = np.argsort(population_fitness)
    
    # Asignar probabilidad basada en el rango (mejor rango = mayor prob)
    ranks = np.arange(pop_size, 0, -1) 
    probabilities = ranks / np.sum(ranks)
    
    # Seleccionar padres usando esas probabilidades
    selected_indices = np.random.choice(ranked_indices, size=num_parents, p=probabilities)
    return selected_indices

def two_point_crossover(parent1, parent2):
    """Cruza en dos puntos."""
    length = len(parent1)
    p1, p2 = np.sort(np.random.choice(range(1, length), 2, replace=False))
    
    child1 = np.concatenate([parent1[:p1], parent2[p1:p2], parent1[p2:]])
    child2 = np.concatenate([parent2[:p1], parent1[p1:p2], parent2[p2:]])
    return child1, child2

def mutate_k_bits(chromosome, k=4):
    """Mutación: Negación de exactamente k bits."""
    length = len(chromosome)
    mutation_indices = np.random.choice(range(length), k, replace=False)
    for idx in mutation_indices:
        chromosome[idx] = 1 - chromosome[idx] # Niega el bit (1 a 0, 0 a 1)
    return chromosome

## Ciclo principal del AG

In [6]:
def run_ga(func_name, dim, max_iter, pop_size=50):
    func = benchmark_funcs[func_name]['func']
    bounds = benchmark_funcs[func_name]['bounds']
    bits_per_var = 8
    chrom_length = dim * bits_per_var
    
    # Inicialización aleatoria binaria
    population = np.random.randint(2, size=(pop_size, chrom_length))
    
    best_overall_fitness = float('inf')
    
    for iteration in range(max_iter):
        # Decodificar y evaluar fitness
        fitness = np.zeros(pop_size)
        for i in range(pop_size):
            real_x = decode_chromosome(population[i], dim, bounds, bits_per_var)
            fitness[i] = func(real_x)
            
        # Reemplazo Generacional: ELITISMO (Guardar el mejor absoluto)
        best_idx = np.argmin(fitness)
        best_parent = population[best_idx].copy()
        if fitness[best_idx] < best_overall_fitness:
            best_overall_fitness = fitness[best_idx]
            
        # Seleccionar padres (N-1 porque 1 espacio es del élite)
        parent_indices = rank_selection(fitness, pop_size - 1)
        
        new_population = [best_parent] # Agregamos al élite primero
        
        # Generar hijos
        for i in range(0, pop_size - 1, 2):
            p1 = population[parent_indices[i]]
            # Si es impar, el último se cruza con uno aleatorio
            p2 = population[parent_indices[i+1]] if i+1 < len(parent_indices) else population[parent_indices[0]]
            
            c1, c2 = two_point_crossover(p1, p2)
            c1 = mutate_k_bits(c1, k=4)
            c2 = mutate_k_bits(c2, k=4)
            
            new_population.append(c1)
            if len(new_population) < pop_size:
                new_population.append(c2)
                
        population = np.array(new_population)
        
    return best_overall_fitness

## PSO con factor de constriccion

In [7]:
def run_pso(func_name, dim, max_iter, pop_size=50):
    func = benchmark_funcs[func_name]['func']
    bounds = benchmark_funcs[func_name]['bounds']
    
    # Parámetros del factor de constricción recomendados estándar (Clerc & Kennedy)
    phi1 = 2.05
    phi2 = 2.05
    phi = phi1 + phi2
    chi = 2.0 / np.abs(2.0 - phi - np.sqrt(phi**2 - 4.0 * phi)) # Aprox 0.7298
    
    # Inicialización de partículas respetando dominios
    positions = np.random.uniform(bounds[0], bounds[1], (pop_size, dim))
    # Velocidades iniciales (generalmente se inician en 0 o pequeñas)
    velocities = np.zeros((pop_size, dim))
    
    # Memoria de las partículas
    pbest_positions = positions.copy()
    pbest_fitness = np.array([func(p) for p in positions])
    
    # Mejor global
    gbest_idx = np.argmin(pbest_fitness)
    gbest_position = pbest_positions[gbest_idx].copy()
    gbest_fitness = pbest_fitness[gbest_idx]
    
    for iteration in range(max_iter):
        for i in range(pop_size):
            r1, r2 = np.random.rand(dim), np.random.rand(dim)
            
            # Actualización de velocidad con factor de constricción
            velocities[i] = chi * (velocities[i] + 
                                   phi1 * r1 * (pbest_positions[i] - positions[i]) + 
                                   phi2 * r2 * (gbest_position - positions[i]))
            
            # Actualización de posición
            positions[i] = positions[i] + velocities[i]
            
            # Limitar posiciones al dominio de la función
            positions[i] = np.clip(positions[i], bounds[0], bounds[1])
            
            # Evaluar nuevo fitness
            current_fitness = func(positions[i])
            
            # Actualizar pbest
            if current_fitness < pbest_fitness[i]:
                pbest_fitness[i] = current_fitness
                pbest_positions[i] = positions[i].copy()
                
                # Actualizar gbest
                if current_fitness < gbest_fitness:
                    gbest_fitness = current_fitness
                    gbest_position = positions[i].copy()
                    
    return gbest_fitness

## Diseño experimental (corridas)

In [9]:
dimensiones = [2, 5, 10, 30, 50]
funciones = ['Sphere', 'Zakharov', 'Rosenbrock', 'Schwefel', 'Michalewicz']
corridas_independientes = 31

resultados_finales = []

print("Iniciando experimentación...")

for func in funciones:
    for dim in dimensiones:
        max_iter = 20 * dim # Regla del documento: 20 iteraciones por dimensión
        print(f"-> Procesando: {func} en {dim}D (Iteraciones: {max_iter})")
        
        resultados_ag = []
        resultados_pso = []
        
        for corrida in range(corridas_independientes):
            ag_best = run_ga(func, dim, max_iter)
            pso_best = run_pso(func, dim, max_iter)
            
            resultados_ag.append(ag_best)
            resultados_pso.append(pso_best)
            
        # Calcular estadísticas para AG
        resultados_finales.append({
            'Función': func, 'Dimensión': dim, 'Algoritmo': 'AG',
            'Mínimo': np.min(resultados_ag),
            'Máximo': np.max(resultados_ag),
            'Media': np.mean(resultados_ag),
            'DE': np.std(resultados_ag),
            'Mediana': np.median(resultados_ag)
        })
        
        # Calcular estadísticas para PSO
        resultados_finales.append({
            'Función': func, 'Dimensión': dim, 'Algoritmo': 'PSO',
            'Mínimo': np.min(resultados_pso),
            'Máximo': np.max(resultados_pso),
            'Media': np.mean(resultados_pso),
            'DE': np.std(resultados_pso),
            'Mediana': np.median(resultados_pso)
        })

print("¡Experimentación finalizada!")

Iniciando experimentación...
-> Procesando: Sphere en 2D (Iteraciones: 40)
-> Procesando: Sphere en 5D (Iteraciones: 100)
-> Procesando: Sphere en 10D (Iteraciones: 200)
-> Procesando: Sphere en 30D (Iteraciones: 600)
-> Procesando: Sphere en 50D (Iteraciones: 1000)
-> Procesando: Zakharov en 2D (Iteraciones: 40)
-> Procesando: Zakharov en 5D (Iteraciones: 100)
-> Procesando: Zakharov en 10D (Iteraciones: 200)
-> Procesando: Zakharov en 30D (Iteraciones: 600)
-> Procesando: Zakharov en 50D (Iteraciones: 1000)
-> Procesando: Rosenbrock en 2D (Iteraciones: 40)
-> Procesando: Rosenbrock en 5D (Iteraciones: 100)
-> Procesando: Rosenbrock en 10D (Iteraciones: 200)
-> Procesando: Rosenbrock en 30D (Iteraciones: 600)
-> Procesando: Rosenbrock en 50D (Iteraciones: 1000)
-> Procesando: Schwefel en 2D (Iteraciones: 40)
-> Procesando: Schwefel en 5D (Iteraciones: 100)
-> Procesando: Schwefel en 10D (Iteraciones: 200)
-> Procesando: Schwefel en 30D (Iteraciones: 600)
-> Procesando: Schwefel en 50D

## Generar tablas de resultados final

In [10]:
# Convertir a DataFrame para visualización limpia
df_resultados = pd.DataFrame(resultados_finales)

# Dar formato para combinar "Media ± DE" como pide la tabla de referencia
df_resultados['Media ± DE'] = df_resultados.apply(
    lambda row: f"{row['Media']:.4e} ± {row['DE']:.4e}", axis=1
)

# Reordenar y limpiar columnas para presentar
columnas_finales = ['Función', 'Dimensión', 'Algoritmo', 'Mínimo', 'Máximo', 'Media ± DE', 'Mediana']
df_reporte = df_resultados[columnas_finales]

# Mostrar la tabla final
display(df_reporte)

# Exportarlo a CSV para pegarlo en tu reporte de Word o Excel
df_reporte.to_csv('resultados_optimizacion.csv', index=False)

,Función,Dimensión,Algoritmo,Mínimo,Máximo,Media ± DE,Mediana
0,Sphere,2,AG,8.062868e-04,3.950805e-02,1.4435e-02 ± 1.2170e-02,1.048173e-02
1,Sphere,2,PSO,2.477397e-08,1.363673e-05,1.4937e-06 ± 2.5494e-06,6.897195e-07
2,Sphere,5,AG,1.169116e-02,1.166294e+00,4.6633e-01 ± 2.8421e-01,4.341854e-01
3,Sphere,5,PSO,1.571905e-09,9.509875e-08,2.4496e-08 ± 2.1479e-08,1.730182e-08
4,Sphere,10,AG,5.748825e-01,2.913114e+00,1.6539e+00 ± 6.1038e-01,1.519851e+00
5,Sphere,10,PSO,6.495307e-12,3.823314e-10,9.5604e-11 ± 9.7902e-11,5.895103e-11
6,Sphere,30,AG,3.811318e+00,8.923176e+00,5.9330e+00 ± 1.1981e+00,5.620625e+00
7,Sphere,30,PSO,1.221837e-13,2.379592e-11,4.4988e-12 ± 5.8192e-12,2.431263e-12
8,Sphere,50,AG,5.028811e+00,1.503644e+01,9.2635e+00 ± 2.1937e+00,9.037669e+00
9,Sphere,50,PSO,4.735528e-12,2.621440e+01,2.5369e+00 ± 7.7503e+00,2.907977e-10
